# weight-decay-l2-add composite — cx12: fold L2 weight-decay into the gradient, then in-place SGD step

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `weight-decay-l2-add`, `inplace-param-update`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "weight-decay-l2-add"
DD_ATOM_IDS = ["weight-decay-l2-add", "inplace-param-update"]
DD_SUBTOPICS = ["Optimizer: Weight decay L2", "PyTorch: In-place param update"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Classical L2 weight-decay is `loss = data_loss + 0.5 * lambda * ||theta||^2`. Its gradient contribution to each param is just `lambda * theta`. The 'coupled L2' implementation (what `torch.optim.SGD(weight_decay=lambda)` does) FOLDS this into the gradient BEFORE the optimizer step:

1. **`weight-decay-l2-add`** — `g <- g + lambda * p`. The grad picks up a 'pull-toward-zero' term proportional to the current parameter. This is in-place on `g` if you own the tensor (but careful — don't mutate `p.grad` itself if you might re-use it elsewhere; here we treat the grad as ours to mutate).
2. **`inplace-param-update`** — `p.data.add_(g, alpha=-lr)`. Same in-place SGD step as before, but `g` now includes the decay term.

**Anatomy.**
```python
for p in params:
    g = p.grad
    if wd != 0:
        g = g + wd * p.data         # weight-decay-l2-add (out-of-place here, common style).
    p.data.add_(g, alpha=-lr)       # inplace-param-update.
```

Both 'in-place on g' (`g.add_(p.data, alpha=wd)`) and 'rebind g locally' work — the param update sees the same effective gradient either way. **Caveat:** if you decide to in-place mutate `p.grad`, the caller's reference now carries the decayed grad too. PyTorch's SGD uses the local-rebind style to avoid surprising the caller.

**Why both atoms together.** L2 is the most commonly enabled SGD knob outside the lr. It only does anything useful if the param update is actually applied — and it has to be computed BEFORE the param update, since it depends on the CURRENT `p`.

### Composite Exercise — fold L2 weight-decay into the gradient, then in-place SGD step

**Atoms exercised together**: `weight-decay-l2-add`, `inplace-param-update`

Implement `cx12_sgd_with_l2(params, grads, lr, weight_decay)`.

For each `(p, g)` pair:

1. If `weight_decay != 0`, compute `g_eff = g + weight_decay * p.data`. (You may either rebind locally OR do `g_eff = g.add(weight_decay * p.data)`; do NOT mutate `p.grad` itself in place — caller might inspect it.)
2. Update the param in place: `p.data.add_(g_eff, alpha=-lr)`. The param's storage must not be reallocated.

Return `None`. Test cross-checks against `torch.optim.SGD(weight_decay=wd, momentum=0)` and verifies:
- The L2 term is applied BEFORE the param step (so it uses the CURRENT `p`, not the post-update one).
- `weight_decay=0` collapses to plain SGD.
- Larger `weight_decay` pulls params harder toward zero across steps.
- Input grads are NOT mutated (caller's reference is preserved).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx12_sgd_with_l2(params, grads, lr, weight_decay):
    """g_eff = g + wd*p; p.data -= lr * g_eff. Returns None."""
    raise NotImplementedError

def _test_cx12():
    t.manual_seed(0)
    shapes = [(3,), (2, 2)]
    params = [t.nn.Parameter(t.randn(s)) for s in shapes]
    ref_params = [t.nn.Parameter(p.data.clone()) for p in params]
    opt_ref = t.optim.SGD(ref_params, lr=0.1, momentum=0.0, weight_decay=0.5)

    param_ptrs_before = [p.data.data_ptr() for p in params]

    # === Step 1 ===
    g1 = [t.randn(s) for s in shapes]
    g1_snapshot = [g.clone() for g in g1]  # to check caller's grads aren't mutated.
    for rp, g in zip(ref_params, g1):
        rp.grad = g.clone()
    opt_ref.step()
    cx12_sgd_with_l2(params, g1, lr=0.1, weight_decay=0.5)

    for i in range(len(params)):
        assert t.allclose(params[i].data, ref_params[i].data, atol=1e-6), (
            f'step 1 params[{i}] mismatch vs torch.optim.SGD(weight_decay=0.5); '
            f'got {params[i].data}, want {ref_params[i].data}'
        )
        assert params[i].data.data_ptr() == param_ptrs_before[i], (
            f'params[{i}].data storage reallocated'
        )
    # Caller's grad tensors must be unchanged.
    for i in range(len(g1)):
        assert t.allclose(g1[i], g1_snapshot[i]), (
            f'grads[{i}] was mutated — do NOT in-place modify the caller\'s grad tensor'
        )

    # === Step 2: verify the L2 uses the CURRENT p (post-step-1), not the original. ===
    g2 = [t.randn(s) for s in shapes]
    for rp, g in zip(ref_params, g2):
        rp.grad = g.clone()
    opt_ref.step()
    cx12_sgd_with_l2(params, g2, lr=0.1, weight_decay=0.5)
    for i in range(len(params)):
        assert t.allclose(params[i].data, ref_params[i].data, atol=1e-6), (
            f'step 2 params[{i}] mismatch — L2 must use the CURRENT p each step'
        )

    # === Case: weight_decay=0 collapses to plain SGD. ===
    p3 = t.nn.Parameter(t.tensor([10.0, -10.0]))
    p3_before = p3.data.clone()
    g3 = [t.tensor([1.0, -1.0])]
    cx12_sgd_with_l2([p3], g3, lr=0.1, weight_decay=0.0)
    assert t.allclose(p3.data, p3_before - 0.1 * g3[0], atol=1e-6), 'wd=0 must be plain SGD'

    # === Case: larger weight_decay pulls params toward zero faster (with zero grad). ===
    p_small_wd = t.nn.Parameter(t.tensor([5.0]))
    p_big_wd   = t.nn.Parameter(t.tensor([5.0]))
    for _ in range(10):
        cx12_sgd_with_l2([p_small_wd], [t.zeros(1)], lr=0.1, weight_decay=0.1)
        cx12_sgd_with_l2([p_big_wd],   [t.zeros(1)], lr=0.1, weight_decay=0.5)
    assert p_big_wd.data.abs().item() < p_small_wd.data.abs().item(), (
        'larger weight_decay should shrink params faster toward zero with grad=0'
    )
    # And both should be strictly between 0 and 5 (decay shrinks; doesn\'t flip sign).
    assert 0.0 < p_small_wd.data.item() < 5.0
    assert 0.0 < p_big_wd.data.item() < 5.0
    _dd_passed.add('cx12')

_test_cx12()

<details><summary>Show solution — cx12</summary>

```python
def cx12_sgd_with_l2(params, grads, lr, weight_decay):
    for p, g in zip(params, grads):
        # Atom A (weight-decay-l2-add): fold L2 into the effective grad using CURRENT p.
        # Out-of-place rebind keeps the caller's g tensor untouched.
        if weight_decay != 0:
            g_eff = g + weight_decay * p.data
        else:
            g_eff = g
        # Atom B (inplace-param-update): step the param in place, storage unchanged.
        p.data.add_(g_eff, alpha=-lr)
    return None
```

Two correctness pitfalls:
- **Order matters.** L2 has to be folded BEFORE the param update, using the CURRENT `p`. Reverse the order and the decay term sees the post-step `p`, which is mathematically a DIFFERENT optimizer (closer to an implicit method).
- **Caller's grad.** PyTorch's SGD does NOT mutate `p.grad` when applying weight_decay — it works on a local copy. We follow that convention here. If your impl does `g.add_(p.data, alpha=weight_decay)`, the caller's grad tensor gets the decay term baked in, which surprises code that wanted to log or post-process raw grads.
This is 'coupled' L2 (a.k.a. classical L2). AdamW does it DIFFERENTLY — see cx21+ for decoupled weight decay.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx12'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx12',
        'subtopics': ["Optimizer: Weight decay L2", "PyTorch: In-place param update"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()